## Delta Lake: Schema Enforcement & Schema Evolution

Delta Lake provides two powerful guardrails for data quality:

| Feature | What it does | Default |
|---|---|---|
| **Schema Enforcement** | Rejects writes whose schema doesn't match the table | Always ON |
| **Schema Evolution** | Allows the table schema to grow (new columns, wider types) | OFF — opt-in per write |

### Why does this matter?
* **Enforcement** prevents silent data corruption — a mistyped column or extra field won't sneak in.
* **Evolution** lets your schema grow naturally (e.g., adding `age` to a users table) without rebuilding.

### What this notebook demonstrates
1. Create a simple Delta table with a fixed schema
2. Attempt to append data with an **extra column** → watch it get rejected
3. Enable **`mergeSchema`** → watch the table evolve gracefully
4. Inspect the schema history via `DESCRIBE HISTORY`

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

TABLE_NAME = "default.schema_demo"

### Step 1 — Create a Delta Table with a Fixed Schema
We start with a simple two-column table: `id` (int) and `name` (string).

In [0]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

data = [(1, "Alice"), (2, "Bob")]
df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print("\u2705 Table created with schema:")
df.printSchema()
display(spark.table(TABLE_NAME))

---
### Step 2 — Schema Enforcement in Action
Now we try to **append** a row that has an extra column (`age`), with `mergeSchema` explicitly **disabled**.
Delta Lake should **reject** this write because the schemas don't match.

```
Existing table:  id (int), name (string)
Incoming data:   id (int), name (string), age (int)   ← extra column!
```

> **Note:** Some Databricks runtimes enable `autoMerge` by default. We use `.option("mergeSchema", "false")` to demonstrate the enforcement behaviour explicitly.

In [0]:
# Build a DataFrame with an extra 'age' column
data_new = [(3, "Charlie", 25)]
schema_new = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
df_new = spark.createDataFrame(data_new, schema_new)

print("Incoming data schema:")
df_new.printSchema()

# Attempt the append with mergeSchema explicitly OFF — this SHOULD fail
try:
    df_new.write.format("delta") \
        .option("mergeSchema", "false") \
        .mode("append") \
        .saveAsTable(TABLE_NAME)
    print("\u274c Unexpected: write succeeded (should have been rejected)")
except Exception as e:
    print("\u2705 Schema Enforcement kicked in! Write was rejected.")
    msg = str(e)
    if "schema mismatch" in msg.lower() or "DELTA_0007" in msg:
        print("\n\U0001f4a1 Reason: A schema mismatch was detected.")
        print("   The table has 2 columns (id, name) but the incoming data has 3 (id, name, age).")
        print('   Hint: use .option("mergeSchema", "true") to allow schema evolution.')

---
### Step 3 — Schema Evolution with `mergeSchema`
Delta lets you **opt in** to schema evolution per write. Just add:
```python
.option("mergeSchema", "true")
```
This tells Delta: *"If the incoming data has new columns, add them to the table schema."*

In [0]:
# Same DataFrame, but now we enable mergeSchema
df_new.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(TABLE_NAME)

print("\u2705 Write succeeded with schema evolution!")
print("\nEvolved schema:")
spark.table(TABLE_NAME).printSchema()

---
### Step 4 — Verify the Evolved Table
Notice how:
* The original rows (Alice, Bob) now have `age = null` — Delta backfills missing columns.
* The new row (Charlie) has all three columns.

In [0]:
display(spark.table(TABLE_NAME))

---
### Step 5 — Inspect Schema History
`DESCRIBE HISTORY` shows every operation on the table, including the schema-evolving write.

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}")
display(history_df.select("version", "timestamp", "operation", "operationParameters"))

---
### Step 6 — Bonus: Type Widening (Schema Evolution for Types)
Delta also supports **widening** column types (e.g., `int` → `long`). Let's demonstrate:

In [0]:
from pyspark.sql.types import LongType

# Data with id as LongType instead of IntegerType
data_wide = [(4, "Diana", 30)]
schema_wide = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
df_wide = spark.createDataFrame(data_wide, schema_wide)

try:
    df_wide.write.format("delta") \
        .option("mergeSchema", "true") \
        .mode("append") \
        .saveAsTable(TABLE_NAME)
    print("\u2705 Type widening succeeded! 'id' column widened from int to long.")
    spark.table(TABLE_NAME).printSchema()
except Exception as e:
    print("\u26a0\ufe0f Type widening was rejected on this runtime:")
    print(f"   {str(e)[:200]}")
    print("\n\U0001f4a1 Note: Automatic type widening requires Delta Lake with the")
    print("   'typeWidening' table feature enabled.")

In [0]:
display(spark.table(TABLE_NAME))

---
### 📝 Key Takeaways

| Concept | How it works |
|---|---|
| **Schema Enforcement** | Automatically rejects writes with mismatched schemas. No config needed — it's always on. |
| **Schema Evolution** | Opt in with `.option("mergeSchema", "true")` per write, or set `spark.databricks.delta.schema.autoMerge.enabled = true` globally. |
| **Backfill** | Existing rows get `null` for newly added columns. |
| **Type Widening** | Narrow types can be widened (e.g., `int` → `long`) with `mergeSchema`. |
| **History** | Every schema change is tracked in `DESCRIBE HISTORY`. |

### Rules of Schema Evolution
* **Allowed**: Adding new columns, widening types (int→long, float→double)
* **Not allowed**: Dropping columns, renaming columns, narrowing types (long→int)
* **Best practice**: Use `mergeSchema` per write rather than the global setting — explicit is safer

In [0]:
# Uncomment to clean up
# spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
# print(f"\U0001f9f9 Dropped {TABLE_NAME}")